# 01 · 데이터 불러오기와 정제  (모듈 2)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/01_load_clean.ipynb)

> **World Bank 개발지표(WDI) 패널**을 GitHub에서 바로 불러와 구조를 보고 정제한다.
> 국가×연도 데이터(2000~2022). 오늘의 원칙: **AI가 코드를 짜고 → 우리가 검증한다.**

지표: 1인당 GDP, 기대수명, 5세 미만 사망률, 인구, 초등 취학률 + 지역·소득그룹.

In [ ]:
# [한 번 실행하고 넘어가세요] 한글 폰트 설정 — Colab 그래프의 한글이 깨지지 않도록(처음 1회 ~20초)
import os, matplotlib.pyplot as plt, matplotlib.font_manager as fm
try:
    p = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"   # 나눔고딕 폰트 경로
    if not os.path.exists(p):                                # 없으면
        os.system("apt-get -qq install -y fonts-nanum > /dev/null 2>&1")  # 설치
    fm.fontManager.addfont(p)                               # matplotlib에 폰트 등록
    plt.rc("font", family="NanumGothic")                   # 기본 글꼴로 지정
except Exception:
    pass                                                    # 실패해도 그냥 진행(영문은 정상)
plt.rc("axes", unicode_minus=False)                         # 마이너스 기호 깨짐 방지

## 0) 불러오기 — GitHub에서 바로 (설치 불필요)

In [ ]:
import pandas as pd, numpy as np    # pandas=표 데이터 다루기, numpy=수치 계산
RAW = "https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/wdi_panel.csv"                                  # GitHub에 올라간 데이터 파일 주소(URL)
df = pd.read_csv(RAW)                          # CSV 파일을 표(DataFrame)로 불러오기
print("행 x 열:", df.shape)                    # (행 수, 열 수) 확인
df.head()                                      # 맨 위 5개 행 미리보기

## 1) 구조 파악 — 컬럼·자료형·결측
진짜 데이터는 **결측이 있다**. 어디에 얼마나 빠졌는지 먼저 본다.

In [ ]:
df.info()   # 각 컬럼의 자료형(숫자/문자)과 '결측 아닌 값' 개수를 한눈에

In [ ]:
df.isna().sum()   # 컬럼별 결측(빈 값) 개수 — under5_mort·prim_enroll에 실제 결측

### 🔎 검증 포인트 — 단위·범위·상식
- 기대수명은 0~90세 범위인가? · 1인당 GDP는 양수인가? · 결측을 어떻게 다룰까?

In [ ]:
# 값이 상식 범위인지 먼저 확인(검증) — AI가 짠 코드라도 사람이 점검한다
print("기대수명 범위:", df.life_exp.min(), "~", df.life_exp.max())   # 0~90세 안이면 정상
print("1인당 GDP 음수:", (df.gdp_pc <= 0).sum(), "건")               # 음수/0이면 이상
df[["gdp_pc","life_exp","under5_mort","pop","prim_enroll"]].describe().round(1)  # 평균·최소·최대 등 요약통계

## 2) 정제 + 파생변수
분석에 쓸 핵심 변수의 결측 행을 제거하고, 금액·인구는 **로그변환**(왜도 큼).

In [ ]:
# 핵심 변수(gdp_pc, life_exp)가 빈 행은 제거. .copy()=원본은 두고 복사본에서 작업
work = df.dropna(subset=["gdp_pc","life_exp"]).copy()
work["log_gdp"] = np.log(work["gdp_pc"])   # 1인당 GDP는 한쪽으로 쏠려(왜도↑) → 로그변환
work["log_pop"] = np.log(work["pop"])      # 인구도 로그변환
print("정제 후 행:", len(work), "/ 원본:", len(df))   # 몇 행이 빠졌는지 확인
work.head(3)

## 3) 빠른 요약 — 소득그룹별 평균 기대수명

In [ ]:
# 소득그룹(income_name)별로 묶어서(group) 기대수명의 '개수'와 '평균' 계산
work.groupby("income_name")["life_exp"]\
    .agg(국가연도수="count", 평균기대수명="mean")\
    .round(1).sort_values("평균기대수명")   # 소수 1자리 반올림 + 평균 오름차순 정렬

## 4) 🖼️ 보기 — 분포·격차를 그림으로
표의 숫자만으론 안 보이는 **쏠림(왜도)**과 **그룹 격차**가 그림엔 바로 드러난다. *왜 로그변환을 했는지*도 그림이 답한다.

In [ ]:
# 왜 로그변환을 했나? — '분포'를 그리면 한눈에 보인다(한쪽으로 쏠림 = 왜도)
import matplotlib.pyplot as plt                                   # 그래프 도구
fig, ax = plt.subplots(1, 2, figsize=(11, 4))                     # 그래프 2개를 가로로
ax[0].hist(work["gdp_pc"], bins=40, color="#bbbbbb", edgecolor="white")    # 원본 1인당 GDP 분포
ax[0].set_title("1인당 GDP (원본) — 오른쪽으로 길게 쏠림")         # 부유한 소수가 긴 꼬리를 만든다
ax[0].set_xlabel("1인당 GDP (US$)"); ax[0].set_ylabel("국가-연도 수")
ax[1].hist(work["log_gdp"], bins=40, color="#4c78a8", edgecolor="white")   # 로그변환 후 분포
ax[1].set_title("log(1인당 GDP) — 종 모양에 가까워짐")             # 쏠림 완화 → 회귀에 더 적합
ax[1].set_xlabel("log(1인당 GDP)"); ax[1].set_ylabel("국가-연도 수")
plt.tight_layout(); plt.show()                                    # 여백 정리 후 표시

In [ ]:
# 소득그룹별 기대수명 — 표의 '평균' 한 숫자를 분포(상자)로 보면 격차가 또렷
order  = ["Low income", "Lower middle income", "Upper middle income", "High income"]  # 낮은→높은 소득
labels = ["저소득", "중하위", "중상위", "고소득"]                                       # 한글 축 이름
data   = [work.loc[work["income_name"] == g, "life_exp"].dropna() for g in order]     # 그룹별 값 묶음
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(data, showmeans=True)                                  # 상자그림(가운데선=중앙값, 삼각=평균)
ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(labels)   # x축에 그룹 이름
ax.set_ylabel("기대수명 (세)"); ax.set_title("소득그룹별 기대수명 — 높을수록 길고 분산도 작다")
plt.tight_layout(); plt.show()

---
✅ **정리**: 불러오기 → 구조·결측 파악 → 검증 → 정제 → 요약. AI에게 시켜도 **이 흐름과 검증은 사람 몫**.

🔁 **폐쇄망 STATA로는?** 같은 작업을 `stata/01_load_clean.do` 에.